# Agent Ranking with Redis Sorted Sets

This notebook demonstrates `RankedOperations` for tracking and ranking agents, tools, and sessions using Redis Sorted Sets.

## Use Cases

- **Agent Performance Ranking** - Track success rates and latencies by task type
- **Tool Effectiveness Ranking** - Find the best-performing tools
- **Session LRU Tracking** - Track recently used sessions for cleanup
- **Token Rate Limiting** - Enforce per-user token budgets

In [ ]:
import os
import random

from redis_openai_agents import RankedOperations

# Configuration
REDIS_URL = os.getenv("REDIS_URL", "redis://localhost:6379")

# Initialize RankedOperations
ranking = RankedOperations(redis_url=REDIS_URL, prefix="demo_rank")
await ranking.initialize()

print(f"Connected to Redis at {REDIS_URL}")

## 1. Agent Performance Ranking

Track agent performance by task type. The score combines:
- Success rate (70% weight)
- Latency factor (30% weight) - faster = higher score

In [ ]:
# Simulate agent performance data for "research" tasks
agents = [
    ("research_agent_1", 0.95, 150),   # High success, fast
    ("research_agent_2", 0.80, 200),   # Medium success, medium speed
    ("research_agent_3", 0.99, 500),   # High success, slower
    ("research_agent_4", 0.60, 100),   # Low success, very fast
]

print("Recording agent performance for 'research' tasks...")
print("=" * 60)

for agent_id, success_rate, avg_latency in agents:
    # Simulate multiple task completions
    for _ in range(10):
        success = random.random() < success_rate
        latency = avg_latency * (0.8 + random.random() * 0.4)  # +/- 20%
        await ranking.record_agent_success(agent_id, "research", success, latency)
    
    print(f"  {agent_id}: ~{success_rate*100:.0f}% success, ~{avg_latency}ms latency")

print("\nDone!")

In [ ]:
# Get top agents for research tasks
top_agents = await ranking.get_best_agents("research", limit=5)

print("Top Agents for 'research' tasks:")
print("=" * 60)
for i, (agent_id, score) in enumerate(top_agents, 1):
    print(f"  {i}. {agent_id}: {score:.4f}")

print("\nNote: Score = 0.7 * success_rate + 0.3 * (1 - latency/10000)")

## 2. Tool Effectiveness Ranking

Track which tools are most effective across all agents.

In [ ]:
# Simulate tool performance data
tools = [
    ("web_search", 0.92, 300),      # High success, moderate speed
    ("calculator", 0.99, 10),       # Very high success, very fast
    ("file_reader", 0.85, 50),      # Good success, fast
    ("code_executor", 0.70, 1000),  # Lower success, slow
    ("db_query", 0.95, 200),        # High success, moderate speed
]

print("Recording tool performance...")
print("=" * 60)

for tool_name, success_rate, avg_latency in tools:
    for _ in range(20):
        success = random.random() < success_rate
        latency = avg_latency * (0.8 + random.random() * 0.4)
        await ranking.record_tool_success(tool_name, success, latency)
    
    print(f"  {tool_name}: ~{success_rate*100:.0f}% success, ~{avg_latency}ms")

print("\nDone!")

In [ ]:
# Get top tools
top_tools = await ranking.get_best_tools(limit=10)

print("Tool Effectiveness Leaderboard:")
print("=" * 60)
for i, (tool_name, score) in enumerate(top_tools, 1):
    print(f"  {i}. {tool_name}: {score:.4f}")

## 3. Session LRU Tracking

Track recently used sessions for memory management and cleanup.

In [ ]:
import time

# Simulate session activity
sessions = ["session_1", "session_2", "session_3", "session_4", "session_5"]

print("Recording session activity...")
print("=" * 60)

for session_id in sessions:
    await ranking.touch_session(session_id)
    print(f"  Touched: {session_id}")
    time.sleep(0.1)  # Small delay between touches

print("\nDone!")

In [ ]:
# Get stale sessions (for demo, use very short max_age)
# In production, you'd use something like 86400 (24 hours)

# First, let's find sessions older than 0.2 seconds
time.sleep(0.3)
stale = await ranking.get_stale_sessions(max_age_seconds=0, limit=10)

print("Sessions that would be candidates for cleanup:")
print("=" * 60)
for session_id in stale:
    print(f"  - {session_id}")

if not stale:
    print("  (No stale sessions found - all recently accessed)")

## 4. Token Rate Limiting

Enforce per-user token budgets using hourly buckets.

In [ ]:
# Simulate token usage for a user with 10,000 token/hour budget
USER_ID = "user_demo_123"
BUDGET_PER_HOUR = 10000

print(f"Token Budget Demo for {USER_ID}")
print(f"Budget: {BUDGET_PER_HOUR:,} tokens/hour")
print("=" * 60)

# Check initial usage
usage = await ranking.get_token_usage(USER_ID, BUDGET_PER_HOUR)
print(f"\nInitial usage: {usage['current_hour']:,} tokens")
print(f"Remaining: {usage['remaining']:,} tokens")

In [ ]:
# Simulate multiple requests consuming tokens
requests = [
    ("Request 1", 1500),
    ("Request 2", 2000),
    ("Request 3", 3000),
    ("Request 4", 2500),
    ("Request 5", 2000),  # This might exceed budget
]

print("Processing requests...")
print("=" * 60)

for name, tokens in requests:
    allowed, remaining = await ranking.check_token_budget(
        USER_ID, tokens, BUDGET_PER_HOUR
    )
    
    status = "ALLOWED" if allowed else "DENIED"
    print(f"  {name}: {tokens:,} tokens - {status}")
    print(f"          Remaining: {remaining:,} tokens")

In [ ]:
# Check final usage
usage = await ranking.get_token_usage(USER_ID, BUDGET_PER_HOUR)

print("\nFinal Token Usage:")
print("=" * 60)
print(f"  Current hour usage: {usage['current_hour']:,} tokens")
print(f"  Remaining budget: {usage['remaining']:,} tokens")
print(f"  Hour bucket: {usage['hour_bucket']}")

## 5. Use Case: Dynamic Agent Selection

Use rankings to select the best agent for a task.

In [ ]:
async def select_best_agent(task_type: str) -> str | None:
    """Select the best-performing agent for a task type."""
    top_agents = await ranking.get_best_agents(task_type, limit=1)
    
    if not top_agents:
        return None
    
    agent_id, score = top_agents[0]
    return agent_id

# Select best agent for research
best_agent = await select_best_agent("research")
print(f"Best agent for 'research' task: {best_agent}")

## Summary

The `RankedOperations` class provides:

1. **Agent Ranking** - Track performance by task type with composite scoring
2. **Tool Ranking** - Find the most effective tools globally
3. **Session LRU** - Track access times for cleanup
4. **Rate Limiting** - Enforce token budgets per user per hour

All operations use Redis Sorted Sets for O(log N) insertions and efficient range queries.

In [ ]:
# Cleanup
await ranking.close()
print("Connection closed.")